# Nemotron Remote Preflight

Run this notebook on **JupyterHub**, **Kaggle** (open via Jupyter URL in Cursor/VS Code), or any Linux GPU kernel.

## How we will use this notebook

- I will update this notebook for you whenever needed.
- You only run the cells I call out.
- If a cell fails, paste the output and I will patch the notebook again.

## Current flow

1. System sanity (`nvidia-smi` / torch); on GPU first `import torch` can take **~1–3 min**
2. **Git clone (CPU + Internet)** — into `/kaggle/working/Nemotron-training` (lost on runtime switch unless you persist, see 2a)
3. **ZIP for Dataset (CPU)** — next cells: build `nemotron-training-for-dataset.zip`, **New Dataset** on Kaggle from that file (**this is what persists**)
4. **Copy from Add data (GPU)** — attach your dataset; copies into `/kaggle/working` when git is unavailable
5. **Verify clone**
6. **Config cell**
7. Deep search (if needed), env, grader tests, …

## Secrets (avoid blocked pushes)

GitHub scans for API keys. Put tokens in **`bootstrap/secrets_local.env`** (gitignored) or in environment / Kaggle secrets — not in committed notebook cells.

## Kaggle notes

- **Where the repo is:** Datasets attached with **Add data** mount under **`/kaggle/input/<dataset-name>/`** (often nested). CONFIG also scans `/kaggle/working`.
- **Secrets — exact labels:** Create secrets whose **name** matches the code (case-sensitive). Prefer: `HF_TOKEN`, `DEEPSEEK_API_KEY`. Also accepted: `HUGGING_FACE_HUB_TOKEN`, `huggingface_token`, `hf_token`, `deepseek_api_key`, `DEEPSEEK_KEY`. Open the **Secrets** panel from this notebook and **attach** each secret to **this** notebook if Kaggle asks.
- If **git clone** fails with `Could not resolve host: github.com`, Internet is off or blocked — use **Input** data or copy under `/kaggle/working`.
- **GPU / runtime changes:** Switching **CPU↔GPU** or starting a **new session** usually provisions a **new VM**, so **`/kaggle/working` is empty** and an old **git clone is gone**. Use a **Kaggle Dataset** (zip of the repo) + **Add data** on every run, then the **copy from Add data** cell (no git, no Internet).


In [ ]:
# Timing: usually seconds. On a cold Kaggle kernel the first `import torch` can take ~1-3 min (I/O + CUDA init).
import os
import sys
import platform
import subprocess


def sh(cmd: str):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(p.stdout.strip() or "(no stdout)")
    if p.stderr.strip():
        print("STDERR:", p.stderr.strip())
    print("-" * 80)
    return p.returncode


print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("-" * 80)

_ = sh("nvidia-smi")

try:
    import torch

    print(
        "torch",
        torch.__version__,
        "cuda",
        torch.cuda.is_available(),
        "gpus",
        torch.cuda.device_count(),
    )
except Exception as e:
    print("torch import failed:", repr(e))

In [ ]:
# OPTIONAL -- clone the repo with git (needs notebook Internet = ON in settings).
# Public repo: default URL works. Private: set env NEMOTRON_GIT_URL to
# https://<token>@github.com/bayntun/Nemotron-training.git
# Shallow clone keeps it fast (~seconds).

from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

DEST = Path("/kaggle/working/Nemotron-training")
URL = os.environ.get(
    "NEMOTRON_GIT_URL", "https://github.com/bayntun/Nemotron-training.git"
)

if (DEST / "eval" / "test_grader.py").is_file():
    print("Repo already present:", DEST)
else:
    if DEST.exists():
        shutil.rmtree(DEST, ignore_errors=True)
    DEST.parent.mkdir(parents=True, exist_ok=True)
    print("Cloning (shallow):", URL)
    print("Destination:", DEST)
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", URL, str(DEST)],
        capture_output=True,
        text=True,
        env=env,
    )
    if proc.returncode != 0:
        err = (proc.stderr or "") + (proc.stdout or "")
        print("\n--- git clone failed (exit", proc.returncode, ") ---")
        print(err.strip() or "(no output)")
        if "Could not resolve host" in err or "Could not resolve" in err:
            print(
                "\n=> Network/DNS blocked. Turn Internet ON in notebook settings "
                "and re-run, OR skip this cell and attach the repo via Add data."
            )
        else:
            print(
                "\n=> See message above. Private repo? Set NEMOTRON_GIT_URL to a token URL."
            )
    else:
        print("Done.")
        if proc.stderr.strip():
            print(proc.stderr.strip())


## CPU-first persistence (Kaggle)

`/kaggle/working` and anything you **`git clone` there are lost** when you switch **CPU to GPU** or start a **new session**.

**Stable approach:** on **CPU + Internet**, clone then run the next cell to build **`nemotron-training-for-dataset.zip`**. Upload that ZIP as a **Kaggle Dataset** (private is fine). On **GPU**, use **Add data** to attach that dataset and run the **copy from Add data** cell, then **CONFIG**.


In [ ]:
# CPU + Internet only: bundle cloned repo into a ZIP you can upload as a Kaggle Dataset.
# Run AFTER git clone (and optional verify). Excludes .git and common caches.

from __future__ import annotations

import zipfile
from pathlib import Path

REPO = Path("/kaggle/working/Nemotron-training")
OUT = Path("/kaggle/working/nemotron-training-for-dataset.zip")
SKIP_PARTS = frozenset(
    {".git", "__pycache__", ".pytest_cache", ".ruff_cache", ".mypy_cache", "venv", ".venv"}
)


def skip_path(p: Path) -> bool:
    return any(x in SKIP_PARTS for x in p.parts)


assert (REPO / "eval" / "test_grader.py").is_file(), (
    f"Clone into {REPO} first (git clone cell), then re-run this cell."
)

if OUT.exists():
    OUT.unlink()

n_files = 0
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in REPO.rglob("*"):
        if not f.is_file() or skip_path(f):
            continue
        arc = Path(REPO.name) / f.relative_to(REPO)
        zf.write(f, arc.as_posix())
        n_files += 1

mib = OUT.stat().st_size / (1024 * 1024)
print("ZIP:", OUT)
print("Files:", n_files, " Size (MiB):", round(mib, 2))
print()
print("1) Download this ZIP from the notebook Files / Output pane.")
print("2) Kaggle -> Datasets -> New Dataset -> upload the ZIP.")
print("3) After you switch to GPU -> Add data -> that dataset -> run 'copy from Add data' -> CONFIG.")


In [ ]:
# Kaggle: quick test — after **Add data**, confirm the repo is visible (expanded, not only a .zip).
from pathlib import Path

MARKER = Path("eval") / "test_grader.py"
KIN = Path("/kaggle/input")


def find_repo_under_dataset(ds: Path, max_depth: int = 6) -> Path | None:
    if not ds.is_dir():
        return None
    frontier = [ds]
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                if (d / MARKER).is_file():
                    return d
            except OSError:
                pass
            try:
                for ch in sorted(d.iterdir()):
                    if ch.is_dir() and not ch.name.startswith("."):
                        nxt.append(ch)
            except OSError:
                pass
        frontier = nxt
        if not frontier:
            break
    return None


print("== Kaggle input mount ==")
if not KIN.is_dir():
    print("No /kaggle/input — not a Kaggle kernel, or path not mounted yet.")
else:
    top = sorted(KIN.iterdir(), key=lambda p: p.name)
    print("Datasets at top level:", [p.name for p in top])
    # Raw `.zip` usually means the upload was not expanded (rare on Kaggle Datasets).
    zips = [p for p in top if p.suffix.lower() == ".zip" and p.is_file()]
    for p in top:
        if p.is_dir():
            try:
                for ch in p.iterdir():
                    if ch.suffix.lower() == ".zip" and ch.is_file():
                        zips.append(ch)
            except OSError:
                pass
    if zips:
        print(".zip paths (may need unzip):", [str(p) for p in zips])

    found: list[Path] = []
    for p in top:
        r = find_repo_under_dataset(p)
        if r is not None:
            found.append(r)

    print("\nRepo root(s) with", MARKER.as_posix(), ":")
    for r in found:
        print(" ", r)
    if not found:
        print("  (none)")

    ok = bool(found)
    print("\nRESULT:", "PASS — ready for the 'copy from Add data' cell." if ok else "FAIL — add the dataset in the sidebar, or unzip if you only see a .zip.")


== Kaggle input mount ==
Datasets at top level: ['competitions', 'datasets', 'models']

Repo root(s) with eval/test_grader.py :
  /kaggle/input/datasets/bayntuna/nemotron-training-bootstrap-code/Nemotron-training

RESULT: PASS — ready for the 'copy from Add data' cell.


In [ ]:
# Kaggle: copy Nemotron-training from **Add data** -> /kaggle/working (writable).
# Use when git/network fails on GPU, or switching CPU<->GPU cleared /kaggle/working.
# One-time: Kaggle Datasets -> New Dataset -> upload ZIP of this repo -> Add data here.

from __future__ import annotations

import shutil
from pathlib import Path

MARKER = Path("eval") / "test_grader.py"
DEST = Path("/kaggle/working/Nemotron-training")


def find_repo_under_input() -> Path | None:
    # Kaggle mounts competition/dataset/model files under /kaggle/input/{competitions,datasets,models}/...
    # with several path segments (e.g. .../datasets/<user>/<slug>/Nemotron-training).
    kin = Path("/kaggle/input")
    if not kin.is_dir():
        return None

    def bfs_repo_under(root: Path, max_depth: int = 8) -> Path | None:
        if not root.is_dir():
            return None
        frontier = [root]
        for _ in range(max_depth):
            nxt = []
            for d in frontier:
                try:
                    if (d / MARKER).is_file():
                        return d
                except OSError:
                    pass
                try:
                    for ch in sorted(d.iterdir()):
                        if ch.is_dir() and not ch.name.startswith("."):
                            nxt.append(ch)
                except OSError:
                    pass
            frontier = nxt
            if not frontier:
                break
        return None

    for top in sorted(kin.iterdir()):
        if not top.is_dir():
            continue
        hit = bfs_repo_under(top)
        if hit is not None:
            return hit
    return None


if (DEST / MARKER).is_file():
    print("Already present:", DEST)
else:
    src = find_repo_under_input()
    if src is None:
        print("No repo found under /kaggle/input with eval/test_grader.py.")
        print("Create a Dataset from a zip of the repo; Add data; re-run this cell.")
    else:
        if DEST.exists():
            shutil.rmtree(DEST, ignore_errors=True)
        shutil.copytree(src, DEST)
        print("Copied:", src, "->", DEST)
        assert (DEST / MARKER).is_file(), "copy incomplete"
        print("OK")



In [ ]:
# VERIFY CLONE — run in the Kaggle **browser** session after the clone cell.
# If /kaggle/working is almost empty: clone was never run in *this* session, or the
# session was restarted (working disk is not permanent across Kaggle runs).

from __future__ import annotations

import os
import subprocess
from pathlib import Path

MARKER = Path("eval") / "test_grader.py"
CANDIDATES = [
    Path("/kaggle/working/Nemotron-training"),
    Path("/kaggle/working/Nemotron"),
    Path.cwd(),
]


def check(label: str, base: Path) -> bool:
    base = base.resolve()
    marker = base / MARKER
    ok = marker.is_file()
    print(f"{label}: {base}")
    print(f"  exists: {base.is_dir()}  has {MARKER}: {ok}")
    return ok


def scan_repo_roots(roots: list[Path], max_depth: int = 14) -> list[Path]:
    """Find directories containing eval/test_grader.py (bounded depth)."""
    found: list[Path] = []
    for root in roots:
        if not root.is_dir():
            continue
        root = root.resolve()
        try:
            for dirpath, dirnames, _filenames in os.walk(
                root, topdown=True, followlinks=False
            ):
                p = Path(dirpath)
                try:
                    depth = len(p.relative_to(root).parts)
                except ValueError:
                    depth = 0
                if depth > max_depth:
                    dirnames[:] = []
                    continue
                dirnames[:] = [d for d in dirnames if not d.startswith(".")]
                if (p / "eval" / "test_grader.py").is_file():
                    found.append(p.resolve())
                    dirnames.clear()
        except OSError as e:
            print(f"[scan warn] {root}: {e}")
    return sorted(set(found))


print("Python cwd:", Path.cwd())
print("-" * 72)

hits: list[Path] = []
for lbl, p in [
    ("default clone path", CANDIDATES[0]),
    ("alt name", CANDIDATES[1]),
    ("cwd", CANDIDATES[2]),
]:
    if check(lbl, p):
        hits.append(p.resolve())

if hits:
    root = hits[0]
    print("-" * 72)
    print("OK: repo root looks like:", root)
    for nm in ("README.md", "pyproject.toml", "requirements.txt"):
        fp = root / nm
        print(f"  {nm}: {fp.is_file()}")
    git = subprocess.run(
        ["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
    )
    print("  git HEAD:", (git.stdout or git.stderr).strip() or "(not a git checkout?)")
else:
    print("-" * 72)
    print("Nothing at the default paths. Scanning /kaggle/working + /kaggle/input ...")
    scan_hits = scan_repo_roots([Path("/kaggle/working"), Path("/kaggle/input")])
    if scan_hits:
        print("Found repo root(s):")
        for h in scan_hits:
            print(" ", h)
    else:
        print("No eval/test_grader.py anywhere under working or input.")
        print()
        print("Most likely: clone was not run in THIS session, or the VM was reset.")
        print("Fix: run the git clone cell again (Internet ON), then re-run this cell.")
    print()
    print("Contents of /kaggle/working:")
    w = Path("/kaggle/working")
    if w.is_dir():
        children = sorted(w.iterdir())
        for ch in children[:50]:
            print(" ", ch)
        if len(children) > 50:
            print("  ...")
    else:
        print("  (missing /kaggle/working)")


In [ ]:
# --- CONFIG (edit this cell only) ---
import os
from pathlib import Path


def _load_kaggle_secrets() -> None:
    """Kaggle Add-ons -> Secrets. Labels must match one of the listed names (case-sensitive)."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return
    sc = UserSecretsClient()

    def pick(*secret_labels: str) -> str:
        for label in secret_labels:
            try:
                v = sc.get_secret(label)
            except Exception:
                v = ""
            if v:
                return v
        return ""

    hf = pick(
        "HF_TOKEN",
        "HUGGING_FACE_HUB_TOKEN",
        "huggingface_token",
        "HUGGINGFACE_TOKEN",
        "hf_token",
    )
    if hf:
        os.environ["HF_TOKEN"] = hf
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf

    ds = pick(
        "DEEPSEEK_API_KEY",
        "deepseek_api_key",
        "DEEPSEEK_KEY",
    )
    if ds:
        os.environ["DEEPSEEK_API_KEY"] = ds

    if os.getenv("HF_TOKEN") and not os.getenv("HUGGING_FACE_HUB_TOKEN"):
        os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]


def _find_first_repo_marker(root: Path, max_depth: int = 5) -> str | None:
    """First directory under root whose eval/test_grader.py exists (BFS)."""
    if not root.is_dir():
        return None
    frontier = [root]
    for _ in range(max_depth):
        nxt: list[Path] = []
        for d in frontier:
            try:
                if (d / "eval" / "test_grader.py").is_file():
                    return str(d)
            except OSError:
                pass
            try:
                for ch in sorted(d.iterdir()):
                    if ch.is_dir() and not ch.name.startswith("."):
                        nxt.append(ch)
            except OSError:
                pass
        frontier = nxt
    return None


_load_kaggle_secrets()


# Auto-pick repo root: /kaggle/input (Input datasets), /kaggle/working, JupyterHub paths.
def _detect_repo_root() -> str:
    candidates = [
        Path("/kaggle/working/Nemotron-training"),
        Path("/kaggle/working/Nemotron"),
        Path("/workspace/nemotron"),
        Path("/home/jovyan/work/Nemotron"),
    ]
    for p in candidates:
        if (p / "eval" / "test_grader.py").exists():
            return str(p)
    kaggle_w = Path("/kaggle/working")
    if kaggle_w.is_dir():
        found = _find_first_repo_marker(kaggle_w, max_depth=4)
        if found:
            return found
    kaggle_in = Path("/kaggle/input")
    if kaggle_in.is_dir():
        for ds in sorted(kaggle_in.iterdir()):
            if not ds.is_dir():
                continue
            found = _find_first_repo_marker(ds, max_depth=6)
            if found:
                return found
    return ""


_detected = _detect_repo_root()
if not _detected and Path("/kaggle/input").is_dir():
    try:
        names = sorted(p.name for p in Path("/kaggle/input").iterdir() if p.is_dir())
        print("Kaggle /kaggle/input datasets:", names)
    except OSError as e:
        print("Could not list /kaggle/input:", e)

NEMOTRON_REPO = _detected

# Default clone/install target if repo not found yet (used by ensure_repo cell).
if not NEMOTRON_REPO:
    if Path("/kaggle/working").is_dir():
        NEMOTRON_REPO = "/kaggle/working/Nemotron-training"
    else:
        NEMOTRON_REPO = "/workspace/nemotron"

# Git repo to clone into NEMOTRON_REPO if the code isn't mounted already.
GITHUB_REPO_URL = "https://github.com/bayntun/Nemotron-training.git"

# If true, will `git clone` into NEMOTRON_REPO when eval/test_grader.py is missing.
AUTO_CLONE_IF_MISSING = True

# If true, runs: pip install -U pip && pip install -r requirements.txt after cloning.
AUTO_INSTALL_DEPS = True

# Secrets — do NOT paste real tokens here (this notebook is often committed).
# Prefer one of:
#   1) JupyterHub / Kaggle secrets / env: HF_TOKEN, DEEPSEEK_API_KEY
#   2) Repo-local file (gitignored): bootstrap/secrets_local.env

if NEMOTRON_REPO:
    os.environ["NEMOTRON_REPO"] = NEMOTRON_REPO

_secrets_base = (
    Path(_detected).expanduser()
    if _detected
    else Path(NEMOTRON_REPO).expanduser()
)
_secrets = _secrets_base / "bootstrap" / "secrets_local.env"
if _secrets.exists():
    for line in _secrets.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key and val:
            os.environ.setdefault(key, val)

# Optional in-cell overrides for one-off experiments only (leave as "" normally).
HF_TOKEN = os.environ.get("HF_TOKEN", "")
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if DEEPSEEK_API_KEY:
    os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY

print("Detected repo (eval/test_grader.py):", _detected or "<none>")
print("Configured NEMOTRON_REPO:", os.getenv("NEMOTRON_REPO", "<unset>"))
print("Repo path exists:", Path(os.getenv("NEMOTRON_REPO", "")).exists())
print("Has eval/test_grader.py:", bool(_detected))
print("secrets_local.env:", "found" if _secrets.exists() else "missing (OK if env vars set)")
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN")))
print("DEEPSEEK_API_KEY set:", bool(os.getenv("DEEPSEEK_API_KEY")))

In [ ]:
# Deep search for the Nemotron repo on Kaggle (or any machine): walks /kaggle/input,
# /kaggle/working, and the current directory for eval/test_grader.py.
#
# Run after CONFIG (secrets are already loaded there). Sets os.environ["NEMOTRON_REPO"]
# and REPO_ROOT to the first match if found.

from __future__ import annotations

import os
from pathlib import Path

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path.cwd(),
]

# Do not walk deeper than this from each root (competition data can be huge).
MAX_DEPTH = 14

SKIP_DIR_NAMES = frozenset(
    {
        ".git",
        "__pycache__",
        "node_modules",
        ".venv",
        "venv",
        ".mypy_cache",
        ".ruff_cache",
        ".pytest_cache",
    }
)


def find_nemotron_repo_dirs() -> list[Path]:
    found: set[Path] = set()
    seen_roots: set[Path] = set()
    for root in SEARCH_ROOTS:
        if not root.is_dir():
            print(f"[skip] not a directory: {root}")
            continue
        root = root.resolve()
        if root in seen_roots:
            continue
        seen_roots.add(root)
        print(f"[walk] {root} (max_depth={MAX_DEPTH})")
        try:
            for dirpath, dirnames, _filenames in os.walk(
                root, topdown=True, followlinks=False
            ):
                p = Path(dirpath)
                try:
                    depth = len(p.relative_to(root).parts)
                except ValueError:
                    depth = 0
                if depth > MAX_DEPTH:
                    dirnames[:] = []
                    continue
                dirnames[:] = [
                    d
                    for d in dirnames
                    if not d.startswith(".") and d not in SKIP_DIR_NAMES
                ]
                if (p / "eval" / "test_grader.py").is_file():
                    found.add(p.resolve())
                    dirnames.clear()
        except (PermissionError, OSError) as e:
            print(f"[warn] walk stopped under {root}: {e}")
    return sorted(found)


candidates = find_nemotron_repo_dirs()
print("\n--- Nemotron repo candidates (contain eval/test_grader.py) ---")
if not candidates:
    print("(none)")
    print(
        "\nAttach the repo with Add data, or copy it under /kaggle/working. "
        "ZIP root should expand to a folder that contains eval/ and test_grader.py."
    )
else:
    for c in candidates:
        print(f"  {c}")
    chosen = candidates[0]
    if len(candidates) > 1:
        print(
            f"\nMultiple matches; using first: {chosen}\n"
            "If wrong, run: os.environ['NEMOTRON_REPO'] = '<path>' manually."
        )
    print("\nSelected:", chosen)
    os.environ["NEMOTRON_REPO"] = str(chosen)
    globals()["REPO_ROOT"] = chosen


In [ ]:
# Persist tokens to repo-local .env (optional). Run after CONFIG + deep search when repo path exists.

from pathlib import Path
import os

repo_root = Path(os.getenv("NEMOTRON_REPO", "")).expanduser()
if "REPO_ROOT" in globals() and REPO_ROOT is not None:
    repo_root = Path(REPO_ROOT)

if not repo_root or not repo_root.is_dir():
    print(
        "Skip .env write: no repo directory yet. Run the deep search cell above first."
    )
elif not (repo_root / "eval" / "test_grader.py").is_file():
    print("Skip .env write:", repo_root, "is not the Nemotron repo root.")
else:
    hf = os.getenv("HF_TOKEN", "") or os.getenv("HUGGING_FACE_HUB_TOKEN", "")
    ds_key = os.getenv("DEEPSEEK_API_KEY", "")
    if not hf or not ds_key:
        print(
            "Skip .env write: missing HF_TOKEN (or HUGGING_FACE_HUB_TOKEN) and/or DEEPSEEK_API_KEY. "
            "On Kaggle, add them in Secrets."
        )
    else:
        env_path = repo_root / ".env"
        env_path.write_text(
            f"HF_TOKEN={hf}\nDEEPSEEK_API_KEY={ds_key}\n",
            encoding="utf-8",
        )
        print(f"Wrote secrets to {env_path}")
        print("NOTE: .env is gitignored in this repo.")

In [ ]:
import os

missing = []
for k in ["HF_TOKEN", "DEEPSEEK_API_KEY"]:
    v = os.getenv(k, "")
    if not v:
        missing.append(k)
    print(k, "SET" if v else "MISSING", f"(len={len(v)})")

if missing:
    print("\nMissing env vars:", ", ".join(missing))
    print(
        "Set them in this notebook session or via JupyterHub secrets/env. "
        "(Do not paste values here; just ensure they are set.)"
    )

In [ ]:
from pathlib import Path
import os
import subprocess


def sh(cmd: str):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(p.stdout.strip() or "(no stdout)")
    if p.stderr.strip():
        print("STDERR:", p.stderr.strip())
    print("-" * 80)
    return p.returncode


def find_repo_root() -> Path | None:
    # Optional override: set NEMOTRON_REPO to the Nemotron repo root.
    env_repo = os.getenv("NEMOTRON_REPO")
    if env_repo:
        p = Path(env_repo).expanduser()
        if (p / "eval" / "test_grader.py").exists():
            return p

    cwd = Path.cwd()

    candidates = [
        cwd,
        cwd / "Nemotron",
        cwd / "Nemotron-training",
        Path("/kaggle/working/Nemotron-training"),
        Path("/kaggle/working/Nemotron"),
        Path("/home/jovyan/work/Nemotron"),
        Path("/home/jovyan/work"),
        Path("/home/jovyan/Nemotron"),
        Path.home() / "Nemotron",
    ]

    for p in candidates:
        if (p / "eval" / "test_grader.py").exists():
            return p

    kaggle = Path("/kaggle/working")
    if kaggle.is_dir():
        for child in sorted(kaggle.iterdir()):
            if child.is_dir() and (child / "eval" / "test_grader.py").exists():
                return child

    kin = Path("/kaggle/input")
    if kin.is_dir():
        for ds in sorted(kin.iterdir()):
            if not ds.is_dir():
                continue
            frontier = [ds]
            for _ in range(6):
                nxt: list[Path] = []
                for d in frontier:
                    try:
                        if (d / "eval" / "test_grader.py").is_file():
                            return d
                    except OSError:
                        pass
                    try:
                        for ch in sorted(d.iterdir()):
                            if ch.is_dir() and not ch.name.startswith("."):
                                nxt.append(ch)
                    except OSError:
                        pass
                frontier = nxt
                if not frontier:
                    break

    return None


_ = sh("pwd")
_ = sh("ls -la")

if "REPO_ROOT" in globals() and REPO_ROOT is not None:
    _rr = Path(REPO_ROOT)
    if (_rr / "eval" / "test_grader.py").is_file():
        repo_root = _rr
        print("repo_root (reusing deep search):", repo_root)
    else:
        repo_root = find_repo_root()
        print("repo_root:", str(repo_root) if repo_root else "MISSING")
else:
    repo_root = find_repo_root()
    print("repo_root:", str(repo_root) if repo_root else "MISSING")

if repo_root is None:
    print("\nERROR: Nemotron repo not found.")
    print(
        "Fix CONFIG (NEMOTRON_REPO), clone into /kaggle/working/Nemotron-training, "
        "or add the repo as Kaggle Input data."
    )
else:
    globals()["REPO_ROOT"] = repo_root
    _git = subprocess.run(
        ["git", "rev-parse", "--is-inside-work-tree"],
        cwd=str(repo_root),
        text=True,
        capture_output=True,
    )
    print("git inside repo:", (_git.stdout.strip() if _git.stdout else "?"))
    _head = subprocess.run(
        ["git", "log", "--oneline", "-1"],
        cwd=str(repo_root),
        text=True,
        capture_output=True,
    )
    print("git HEAD:", _head.stdout.strip() if _head.stdout else "<none>")


In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def ensure_repo_root() -> Path:
    if "REPO_ROOT" in globals() and REPO_ROOT is not None:
        return Path(REPO_ROOT)

    target = os.getenv("NEMOTRON_REPO", "")
    if not target:
        raise RuntimeError("REPO_ROOT missing and NEMOTRON_REPO is unset. Set CONFIG cell." )

    target_path = Path(target).expanduser()
    if (target_path / "eval" / "test_grader.py").exists():
        globals()["REPO_ROOT"] = target_path
        return target_path

    auto_clone = globals().get("AUTO_CLONE_IF_MISSING", True)
    github_url = globals().get("GITHUB_REPO_URL", "")
    if not auto_clone:
        raise RuntimeError(f"Repo missing at {target_path} and AUTO_CLONE_IF_MISSING=false")
    if not github_url:
        raise RuntimeError("Repo missing and GITHUB_REPO_URL is empty.")

    print(f"\nRepo missing at {target_path}. Cloning...")
    target_path.parent.mkdir(parents=True, exist_ok=True)
    if target_path.exists():
        subprocess.run(["rm", "-rf", str(target_path)], check=False)

    subprocess.run(["git", "clone", github_url, str(target_path)], check=True)

    auto_install = globals().get("AUTO_INSTALL_DEPS", False)
    if auto_install:
        print("Installing requirements.txt...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-U", "pip"],
            check=True,
            text=True,
        )
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-r",
                str(target_path / "requirements.txt"),
            ],
            check=True,
            text=True,
        )

    if not (target_path / "eval" / "test_grader.py").exists():
        raise RuntimeError("Clone completed but eval/test_grader.py still missing.")

    globals()["REPO_ROOT"] = target_path
    return target_path


repo_root = ensure_repo_root()
print("Using REPO_ROOT:", repo_root)

cmd = [sys.executable, "-m", "pytest", "eval/test_grader.py", "-q"]
p = subprocess.run(cmd, cwd=str(repo_root), text=True, capture_output=True)
print(p.stdout)
print(p.stderr)
print("exit_code:", p.returncode)

In [ ]:
import sys, subprocess

assert "REPO_ROOT" in globals(), "Run repo discovery cell first (it sets REPO_ROOT)."
assert os.getenv("HF_TOKEN"), "HF_TOKEN missing. Set it in CONFIG cell or JupyterHub env."

cmd = [sys.executable, "-m", "data.download", "--sft-only"]
p = subprocess.run(cmd, cwd=str(REPO_ROOT), text=True, capture_output=True)
print(p.stdout[-4000:])
print(p.stderr[-2000:])
print("exit_code:", p.returncode)

In [ ]:
import sys, subprocess

assert "REPO_ROOT" in globals(), "Run repo discovery cell first (it sets REPO_ROOT)."
assert os.getenv("DEEPSEEK_API_KEY"), "DEEPSEEK_API_KEY missing. Set it in CONFIG cell or JupyterHub env."

cmd = [sys.executable, "-m", "teacher.smoke_test"]
p = subprocess.run(cmd, cwd=str(REPO_ROOT), text=True, capture_output=True)
print(p.stdout[-4000:])
print(p.stderr[-2000:])
print("exit_code:", p.returncode)